# 'Equity' project

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os

# # Search for your file
# for root, dirs, files in os.walk('/content/drive/MyDrive'):
#     for file in files:
#         if 'equity_contexts' in file.lower():
#             print(os.path.join(root, file))

In [2]:
import pandas as pd

In [3]:
!pip install -q transformers torch

In [4]:
# Load full corpus and filter to equity sentences only
df_full = pd.read_csv("01_data/full_sentence_corpus.csv")
df = df_full[df_full["contains_equity"] == True].copy().reset_index(drop=True)

# Reconstruct ±1 sentence context window
df["context_window"] = (
    df["prev_sentence"].fillna("")
    + " "
    + df["sentence_text"]
    + " "
    + df["next_sentence"].fillna("")
).str.strip()

print(f"Total equity sentences: {len(df):,}")
print(df["category"].value_counts())

Total equity sentences: 4,060
category
state_local        2202
federal_policy     1036
ngo_nonprofit       630
academic            140
news_commentary      52
Name: count, dtype: int64


In [5]:
# Load the BERT model
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"Using device: {device}")
print("BERT loaded.")

/Users/sebinescaria/miniforge3/envs/equity_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8804.13it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when load

Using device: cpu
BERT loaded.


In [6]:
# Generate the embeddings
from tqdm import tqdm


def get_equity_embedding(text, tokenizer, model, device, max_length=256):
    text_lower = text.lower()
    if "equity" not in text_lower:
        return None

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True,
        padding="max_length",
    )

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    equity_positions = [i for i, t in enumerate(tokens) if "equity" in t.lower()]

    if not equity_positions:
        return None

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    hidden_states = outputs.hidden_states
    last_4 = torch.stack(hidden_states[-4:])
    avg_last_4 = last_4.mean(dim=0)

    equity_embeddings = [
        avg_last_4[0, pos, :].cpu().numpy() for pos in equity_positions
    ]
    return np.mean(equity_embeddings, axis=0)


embeddings = []
valid_indices = []

print(f"Generating embeddings for {len(df):,} equity sentences...")

for i in tqdm(range(len(df))):
    text = str(df["context_window"].iloc[i])
    emb = get_equity_embedding(text, tokenizer, model, device)
    if emb is not None:
        embeddings.append(emb)
        valid_indices.append(i)

embeddings = np.array(embeddings)
df_valid = df.iloc[valid_indices].reset_index(drop=True)

print(f"Embeddings generated: {len(embeddings):,}")
print(f"Embedding dimension: {embeddings.shape[1]}")

# Save to Drive
np.save("01_data/equity_embeddings_v2.npy", embeddings)
df_valid.to_csv("01_data/equity_embeddings_meta_v2.csv", index=False)
print("Saved to Drive.")

Generating embeddings for 4,060 equity sentences...


100%|██████████| 4060/4060 [06:18<00:00, 10.71it/s]

Embeddings generated: 4,040
Embedding dimension: 768
Saved to Drive.


In [7]:
# K-Means
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

# Normalize embeddings
embeddings_norm = normalize(embeddings)

# Test k=2 to k=5 and pick best silhouette score
print("Selecting optimal k by silhouette score...\n")
silhouette_scores = {}

for k in range(2, 6):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings_norm)
    score = silhouette_score(embeddings_norm, labels, sample_size=2000, random_state=42)
    silhouette_scores[k] = score
    print(f"  k={k}: silhouette score = {score:.4f}")

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nBest k: {best_k} (score: {silhouette_scores[best_k]:.4f})")

# Fit final model with best k
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(embeddings_norm)
df_valid["cluster"] = cluster_labels

print("\nCluster distribution:")
print(df_valid["cluster"].value_counts().sort_index())
print("\nCluster distribution by document type:")
print(pd.crosstab(df_valid["cluster"], df_valid["category"]))

Selecting optimal k by silhouette score...

  k=2: silhouette score = 0.0922
  k=3: silhouette score = 0.0825
  k=4: silhouette score = 0.0773
  k=5: silhouette score = 0.0350

Best k: 2 (score: 0.0922)

Cluster distribution:
cluster
0    2331
1    1709
Name: count, dtype: int64

Cluster distribution by document type:
category  academic  federal_policy  news_commentary  ngo_nonprofit  \
cluster                                                              
0               80             651               23            294   
1               60             381               29            330   

category  state_local  
cluster                
0                1283  
1                 909  


In [8]:
print("=== Representative Examples per Cluster ===\n")

for cluster_id in sorted(df_valid["cluster"].unique()):
    cluster_data = df_valid[df_valid["cluster"] == cluster_id]
    print(f"--- Cluster {cluster_id} ({len(cluster_data):,} contexts) ---")
    print("Document type breakdown:")
    print(cluster_data["category"].value_counts().to_string())
    print("\nSample contexts:")
    samples = (
        cluster_data["sentence_text"]
        .sample(min(3, len(cluster_data)), random_state=42)
        .values
    )
    for j, s in enumerate(samples, 1):
        print(f"  [{j}] {s[:300]}...")
    print()

=== Representative Examples per Cluster ===

--- Cluster 0 (2,331 contexts) ---
Document type breakdown:
category
state_local        1283
federal_policy      651
ngo_nonprofit       294
academic             80
news_commentary      23

Sample contexts:
  [1] " The social determinants of health can be grouped into 5 domains: • Economic stability • Education access and quality • Health care access and quality • Neighborhood and built environment • Social and community context Health Equity and Health Disparities Environmental Scan 32 Appendix B: Health Eq...
  [2] There remain fundamental questions about the data used to calculate health equity measures, with regard to completeness, accuracy and assumptions about underlying categories (e.g., risk of labels for race and ethnicity perpetuating racist structures)....
  [3] SHIP Health Equity Social Determinants of Policy and Systems LHIC Aligned Place-Based Cross-Sector 53 Women's Health Objective 3.2.1 Goal 2: Increase breast and cervical c

In [9]:
tableau_df = df_valid[
    [
        "sentence_text",
        "category",
        "source_label",
        "title",
        "document_subtype",
        "is_definition_sentence",
        "contains_health_equity",
        "word_count",
        "cluster",
    ]
].copy()

tableau_df.to_csv("01_data/equity_tableau.csv", index=False)
print(f"Exported {len(tableau_df):,} rows for Tableau.")
print("Download equity_tableau.csv from your Drive and load it into Tableau.")

Exported 4,040 rows for Tableau.
Download equity_tableau.csv from your Drive and load it into Tableau.
